# 01. NumPy 기초
**목표**: 반복문 없이 배열을 한 번에 계산하는 **벡터화**를 익히고, 이미지 여러 장의 평균·표준편차를 구해 봅니다. (예상 6~8시간)

> 미션 4.3 "이미지의 평균과 표준편차를 for문 없이 계산" 의 기초입니다.

## 왜 NumPy인가?
디지털 이미지는 **숫자가 담긴 격자(배열)** 입니다. 흑백 28×28 이미지는 0(검정)~255(흰색)의 숫자 784개입니다.

In [2]:
import numpy as np

## 1. 배열 만들기와 기본 속성

In [3]:
a = np.array([1, 2, 3, 4, 5])
print(a)
print("shape:", a.shape)   # 모양
print("dtype:", a.dtype)   # 원소 자료형
print("ndim :", a.ndim)    # 차원 수

[1 2 3 4 5]
shape: (5,)
dtype: int64
ndim : 1


In [4]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])
print(m)
print("shape:", m.shape)   # (행 수, 열 수)

print(np.zeros((2, 3)))
print(np.ones((2, 3)))
print(np.arange(0, 10, 2))
print(np.random.default_rng(0).random((2, 3)))   # 0~1 난수

[[1 2 3]
 [4 5 6]]
shape: (2, 3)
[[0. 0. 0.]
 [0. 0. 0.]]
[[1. 1. 1.]
 [1. 1. 1.]]
[0 2 4 6 8]
[[0.63696169 0.26978671 0.04097352]
 [0.01652764 0.81327024 0.91275558]]


## 2. 인덱싱과 슬라이싱

In [5]:
img = np.arange(36).reshape(6, 6)   # 6x6 작은 '이미지'
print(img)
print("한 픽셀 img[2, 3]    :", img[2, 3])
print("두 번째 행            :", img[1])
print("두 번째 열            :", img[:, 1])
print("좌측 상단 3x3 영역    :\n", img[:3, :3])

[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]
 [12 13 14 15 16 17]
 [18 19 20 21 22 23]
 [24 25 26 27 28 29]
 [30 31 32 33 34 35]]
한 픽셀 img[2, 3]    : 15
두 번째 행            : [ 6  7  8  9 10 11]
두 번째 열            : [ 1  7 13 19 25 31]
좌측 상단 3x3 영역    :
 [[ 0  1  2]
 [ 6  7  8]
 [12 13 14]]


### 다운샘플링 (미션 허용 방식)
`arr[::n, ::n]` 은 행과 열을 n칸씩 건너뛰어 이미지를 **작게** 만듭니다.

In [8]:
small = img[::2, ::2]
print(small.shape)
print(small)

(3, 3)
[[ 0  2  4]
 [12 14 16]
 [24 26 28]]


## 3. 벡터화 연산 ⭐
같은 일을 **for문** 과 **벡터 연산** 으로 해 보고 비교합니다.

In [9]:
import time
x = np.arange(1_000_000)

# (1) for문
t0 = time.perf_counter()
result_loop = [v * 2 for v in x]
t1 = time.perf_counter()

# (2) 벡터화
result_vec = x * 2
t2 = time.perf_counter()

print(f"for문   : {t1 - t0:.4f}초")
print(f"벡터화  : {t2 - t1:.4f}초")

for문   : 0.0662초
벡터화  : 0.0011초


산술 연산, 비교 연산이 모두 **원소별로 한 번에** 적용됩니다.

In [10]:
v = np.array([10, 20, 30, 40])
print(v + 5)
print(v * v)
print(v > 25)          # 불리언 배열
print(v[v > 25])       # 불리언 인덱싱: 조건에 맞는 것만 꺼내기

[15 25 35 45]
[ 100  400  900 1600]
[False False  True  True]
[30 40]


백터화 연산을 하는 이유: numpy가 속도가 빠르기 때문.

## 4. 집계와 `axis` ⭐
`mean`, `std`, `sum`, `min`, `max` 등은 `axis` 로 **어느 방향으로 합칠지** 정합니다.

- `axis=0`: 행 방향으로 합침 → 열별 결과
- `axis=1`: 열 방향으로 합침 → 행별 결과

In [11]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])
print("전체 평균   :", m.mean())
print("열별 평균   :", m.mean(axis=0))   # [2.5 3.5 4.5]
print("행별 평균   :", m.mean(axis=1))   # [2. 5.]

전체 평균   : 3.5
열별 평균   : [2.5 3.5 4.5]
행별 평균   : [2. 5.]


## 5. 이미지 여러 장 다루기 (3차원 배열)
이미지 N장을 쌓으면 `(N, 높이, 너비)` 모양의 3차원 배열이 됩니다.
이미지 **한 장 안에서** 평균을 내려면 높이·너비 두 축을 함께 지정합니다: `axis=(1, 2)`.

In [12]:
rng = np.random.default_rng(42)
images = rng.integers(0, 256, size=(100, 28, 28))   # 100장, 28x28, 0~255
print("shape:", images.shape)

img_mean = images.mean(axis=(1, 2))   # 이미지별 평균 → 길이 100
img_std  = images.std(axis=(1, 2))    # 이미지별 표준편차 → 길이 100

print("평균 shape:", img_mean.shape)
print("첫 3장 평균:", img_mean[:3].round(2))
print("첫 3장 표준편차:", img_std[:3].round(2))

shape: (100, 28, 28)
평균 shape: (100,)
첫 3장 평균: [129.51 127.17 126.01]
첫 3장 표준편차: [73.59 74.48 73.5 ]


**for문으로 검증**해서 결과가 같은지 확인합니다. (실제 미션 코드에서는 for문을 쓰지 않습니다.)

In [13]:
check = np.array([im.mean() for im in images])
print("두 방법 결과 동일?", np.allclose(img_mean, check))

두 방법 결과 동일? True


### 이 값이 의미하는 것
- **평균**: 이미지의 전반적인 밝기 (클수록 밝은 이미지)
- **표준편차**: 명암 대비 (클수록 밝고 어두운 부분 차이가 큼)

상품 이미지에서 이 두 값을 구해 새 컬럼으로 붙이는 것이 미션 4.3입니다.

## 6. CSV에 문자열로 저장된 배열 복원
CSV에는 배열이 `"[[1, 2], [3, 4]]"` 같은 문자열로 저장될 수 있습니다. `ast.literal_eval` 로 복원합니다. (미션 허용 방식)

In [14]:
import ast

s = "[[0, 128], [255, 64]]"
arr = np.array(ast.literal_eval(s))
print(arr, arr.shape, arr.mean())

[[  0 128]
 [255  64]] (2, 2) 111.75


## 7. 정리

| 기억할 것 | 예 |
|---|---|
| 반복문 대신 배열 연산 | `x * 2`, `x > 3` |
| `axis` 로 집계 방향 지정 | `images.mean(axis=(1, 2))` |
| 슬라이싱으로 다운샘플링 | `img[::2, ::2]` |
| 문자열 → 배열 | `np.array(ast.literal_eval(s))` |

### ✏️ 직접 해보기
`(50, 16, 16)` 모양의 랜덤 이미지(0~255)를 만들고, 각 이미지의 **평균이 가장 큰 이미지의 번호**를 구하세요. 이어서 평균이 128보다 큰 이미지가 몇 장인지도 구하세요.

> 힌트: `argmax`, 불리언 배열의 `sum`

In [ ]:
# 여기에 직접 작성해 보세요

**정답 예시** (먼저 직접 풀어본 뒤 확인하세요)

In [15]:
rng = np.random.default_rng(1)
imgs = rng.integers(0, 256, size=(50, 16, 16))
means = imgs.mean(axis=(1, 2))
print("가장 밝은 이미지 번호:", means.argmax())
print("평균 128 초과 장수   :", (means > 128).sum())

가장 밝은 이미지 번호: 27
평균 128 초과 장수   : 30


### ✏️ 직접 해보기
위 `imgs` 를 `[:, ::2, ::2]` 로 다운샘플링하면 shape이 어떻게 되는지 예측하고 확인하세요.

In [ ]:
# 여기에 직접 작성해 보세요

**정답 예시** (먼저 직접 풀어본 뒤 확인하세요)

In [16]:
print(imgs[:, ::2, ::2].shape)   # (50, 8, 8)

(50, 8, 8)


✅ 다음: `02_pandas_basics`